In [ ]:
# MLP 是 Multilayer Perceptron（多层感知机）

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import shap # 借用 shap 的数据集

X, y = shap.datasets.adult()

# 神经网络对类别非常敏感，这里简单处理：只取数值列进行演示，或者将所有列转为数值
# 为了演示方便，我们选几个数值列
numeric_cols = ['Age', 'Education-Num', 'Capital Gain', 'Capital Loss', 'Hours per week']
X_sub = X[numeric_cols].values
y_sub = y.astype(float)

# 【关键】标准化：神经网络必须做这一步
scaler = StandardScaler()   #神经网络中，每个特征都会乘以一个权重，需标准化
X_scaled = scaler.fit_transform(X_sub)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_sub, test_size=0.2, random_state=42)

# 【关键】转换为 PyTorch 张量 (Tensor)
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).view(-1, 1) # 形状变为 (N, 1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).view(-1, 1)

# 2. 定义神经网络架构 (MLP)
class IncomeMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__() # 调用父类构造函数
        # 定义层结构：输入层 -> 隐藏层1 -> 隐藏层2 -> 输出层
        self.layer1 = nn.Linear(input_dim, 64)  # 线性连接y=w*x+b # 输入维度 -> 64个神经元
        self.layer2 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)
        self.relu = nn.ReLU()                  # 激活函数 # max(0, x)
        self.sigmoid = nn.Sigmoid()            # 将输出(-∞, +∞)压缩到 0-1 (概率)

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.sigmoid(self.output(x))
        return x

# 3. 初始化模型、损失函数和优化器
model = IncomeMLP(input_dim=len(numeric_cols))
criterion = nn.BCELoss() #Binary Cross Entropy Loss 二分类交叉熵损失
optimizer = optim.Adam(model.parameters(), lr=0.001) # Adam 优化器  Adaptive Moment Estimation自适应调节

# 4. 训练循环 (Epochs)
epochs = 50
for epoch in range(epochs):
    model.train() # 设置为训练模式
    
    # --- 标准五步法 ---
    # 1. 前向传播
    outputs = model(X_train_t) #调用forward
    loss = criterion(outputs, y_train_t)
    
    # 2. 梯度清零
    optimizer.zero_grad()
    
    # 3. 反向传播 (计算梯度)
    loss.backward()
    
    # 4. 更新权重
    optimizer.step()
    
    # 每 10 轮打印一次
    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# 5. 模型评估
model.eval() # 设置为评估模式
with torch.no_grad(): # 评估时不需要计算梯度，节省显存
    test_outputs = model(X_test_t)
    predicted = (test_outputs > 0.5).float()
    accuracy = (predicted == y_test_t).sum() / y_test_t.shape[0]
    print(f"\n测试集准确率: {accuracy.item():.4f}")

Epoch [10/50], Loss: 0.6014
Epoch [20/50], Loss: 0.5437
Epoch [30/50], Loss: 0.4917
Epoch [40/50], Loss: 0.4520
Epoch [50/50], Loss: 0.4269

测试集准确率: 0.8029


### 为什么神经网络中间层推荐使用 ReLU 而非 Sigmoid？

#### 1. Sigmoid 的“致命伤”：梯度消失 (Vanishing Gradient)
* **问题核心：** 当输入值非常大或非常小时，Sigmoid 函数的导数（坡度）趋近于 0。
* **连锁反应：** 在反向传播过程中，多个接近 0 的导数连乘，会导致梯度极速衰减。
* **最终后果：** 网络靠近输入端的权重几乎无法更新，导致模型出现“练不动”的情况。

#### 2. ReLU 的核心优势
* **梯度恒定：** 对于 $x > 0$ 的部分，其导数恒为 1。这确保了在深层网络中，梯度（信号）可以有效回传，避免了梯度消失。
* **计算高效：** ReLU 的计算仅涉及简单的正负判断（$f(x) = \max(0, x)$），无需像 Sigmoid 那样计算昂贵的指数运算 $e^x$。
* **稀疏性：** 通过将部分神经元输出设为 0，ReLU 产生了稀疏激活效果，这在一定程度上模拟了生物神经元的稀疏激活特性，有助于提高模型的表示能力。

### 多分类问题的输出层
* **激活函数：** Softmax
* **逻辑：** 将原始得分（Logits）转化为概率分布，确保输出 $\sum P_i = 1$。
* **搭配使用：** 通常与交叉熵损失函数（Cross-Entropy Loss）共同用于多分类任务。